In [1]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_excel("../muestra_terra/Terra_example.xlsx")

In [24]:
import re
from collections import Counter

# Asegúrate de tener cargado tu DataFrame como `df`
requests = df['Request'].dropna().astype(str).tolist()

# Patrones para detectar secciones dentro del texto
section_patterns = [
    r"in the ([A-Z\s]+?) section",
    r"in the ([A-Z\s]+?) page",
    r"on the ([A-Z\s]+?) section",
    r"on the ([A-Z\s]+?) page",
    r"in ([A-Z\s]+?) section",
    r"on ([A-Z\s]+?) page"
]

# Extraer y normalizar
found_sections = []
for text in requests:
    for pattern in section_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        found_sections.extend([m.strip().title() for m in matches])

# Contar ocurrencias
section_counts = Counter(found_sections)
print(section_counts.most_common(10))

[('Who We Are', 5), ('The Who We Are', 4), ('The', 1), ('Keeping The Header Copy The Same On Both', 1)]


In [9]:
import pandas as pd
import random
from faker import Faker
from transformers import pipeline, set_seed
from datetime import date, datetime, timedelta

# Inicialización
fake = Faker()
set_seed(42)
generator = pipeline("text-generation", model="gpt2")

# Valores reales de Request Type y demás categorías
request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
sections = ['hero section', 'footer', 'about us page', 'contact form', 'pricing table', 'FAQ block']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]  # pesos para fechas de 2025 o posteriores

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

# Función para generar texto

introductions = [
    "Hey team, this is {requester} from {company}.",
    "Hi! {requester} here (from {company})",
    "Hola, soy {requester}, de {company}.",
    "Hey! Working on the {project}, I noticed the following:",
    "{requester} from {company}, reporting an issue on {page}",
    "Hi again! Quick one from {requester} at {company}.",
    "Ey, soy {requester}, lo vi mientras revisaba el {project}.",
    "From {company} – {requester} noticed something in the UI.",
    "Hi, this is about the {project}. I’m {requester} from {company}."
]

tech_terms_by_type = {
    'Copy Revision': ['tone of voice', 'CTA', 'microcopy', 'UX writing'],
    'Design Issues': ['UI consistency', 'alignment', 'spacing', 'responsive layout', 'mobile-first'],
    'Requested Change': ['client feedback', 'scope update', 'functional change', 'new spec'],
    'New Item': ['feature', 'widget', 'module', 'custom component', 'new interaction']
}

def generate_request(request_type, section, priority, company, requester, project, page):
    # Variaciones en cómo se nombra la sección
    section_variants = [
        f"the {section}",
        f"section '{section}'",
        f"'{section}' section",
        f"area: {section}",
        f"in {section}"
    ]

    urgency_phrases = {
        'Low': [
            "", "", "", 
            " Not urgent.",
            " Can be handled later."
        ],
        'Normal': [
            " Please address this soon.",
            " Handle this in the upcoming days.",
            " This should be scheduled accordingly."
        ],
        'High': [
            " This is a high priority item.",
            " Should be prioritized.",
            " Requires prompt attention."
        ],
        'Urgent': [
            " URGENT! This blocks progress.",
            " Immediate action required.",
            " Needs to be resolved ASAP.",
            " Critical path — fix now!"
        ]
    }

    templates = {
        'Copy Revision': [
            "Please revise the copy in {section}.",
            "Copy update needed for {section}.",
            "Text correction required in {section}.",
            "Rewriting needed for {section}.",
            "The content in {section} needs improvement."
        ],
        'Design Issues': [
            "Design problem found in {section}.",
            "Layout issue detected in {section}.",
            "Inconsistency in design of {section}.",
            "Visual bug in {section} needs fixing.",
            "Improve visual design in {section}."
        ],
        'Requested Change': [
            "Client requested a change in {section}.",
            "Requested modification for {section}.",
            "Adjustment needed in {section}.",
            "Update required following feedback in {section}.",
            "Changes needed according to client input in {section}."
        ],
        'New Item': [
            "New component needed in {section}.",
            "Add a new feature to {section}.",
            "Create something new in {section}.",
            "New item requested for {section}.",
            "Build a new section in {section}."
        ]
    }

    tech_terms = tech_terms_by_type.get(request_type, [])
    buzzword = random.choice(tech_terms) if tech_terms and random.random() < 0.5 else ""

    section_phrase = random.choice(section_variants)
    base_template = random.choice(templates.get(request_type, ["Update {section}."]))
    base = base_template.replace("{section}", section_phrase)

    if buzzword:
        base += f" Consider {buzzword}."

    urgency = random.choice(urgency_phrases.get(priority, [""]))
    punctuation = random.choice([".", "!", "..."])

    intro = ""
    if random.random() < 0.7:  # 70% chance de añadir intro
        intro_template = random.choice(introductions)
        intro = intro_template.format(
            requester=requester,
            company=company,
            project=project,
            page=page
        ) + " "

    return intro + base + urgency + punctuation

# Tema clientes y tamaños
num_clients = 25
client_names = list({fake.company() for _ in range(num_clients * 2)})[:num_clients]
random.shuffle(client_names)
clients = {}
for i, name in enumerate(client_names):
    if i < 5:
        clients[name] = 'large'
    elif i < 15:
        clients[name] = 'medium'
    else:
        clients[name] = 'small'
size_weights = {'large': 5, 'medium': 3, 'small': 1}
weighted_clients = []
for client, size in clients.items():
    weighted_clients.extend([client] * size_weights[size])

# Generación de proyectos por cliente con lógica según tamaño
project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
projects_per_client = {}
for client, size in clients.items():
    num_projects = {
        'large': random.randint(5, 8),
        'medium': random.randint(3, 5),
        'small': random.randint(1, 2)
    }[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        project_name = f"{base} {suffix} – {color}"
        projects.add(project_name)
    projects_per_client[client] = list(projects)

# Fechas base por proyecto
start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)
project_date_bases = {}
for project_list in projects_per_client.values():
    for project in project_list:
        project_date_bases[project] = fake.date_between(start_date=start_date, end_date=end_date)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

# Browser
weighted_browsers = (
    ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 +
    ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
)

# Devices
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

# Generador de página web
def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

#Cosas de tiempo
estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

# Crear datos sintéticos
data = []
for _ in range(10000):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)
    priority = random.choice(priority_weights_by_type[request_type])
    page = generate_client_page(project)

    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    real_time = max(1, round(random.normalvariate(estimated_time, 0.3)))
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    # Status
    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]


        # Calcular real_time según estado
    if status_value == "Complete":
        real_time = max(1, round(random.normalvariate(estimated_time, estimated_time * 0.3)))
    elif status_value == "In Progress":
        real_time = round(estimated_time * random.uniform(1.0, 1.5))
    else:  # In Queue
        real_time = 0

    requester = fake.name()


    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": requester,
        "Request Type": request_type,
        "Priority": priority,
        "Request": generate_request(request_type, section, priority, client, requester, project, page),
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })

# Convertir a DataFrame
df = pd.DataFrame(data)

Device set to use cpu


In [8]:
df

,Company,Project Name,Input Date,Status,Requester,Request Type,Priority,Request,Device,Browser,Page,Estimated Time (tokens),Real Time
0,Conway PLC,Detail Revamp – OliveDrab,16/06/2022,Complete,Cynthia Todd,New Item,High,New component needed in the footer Requires pr...,Desktop,Chrome,https://www.detailrevamp–olivedrab.com,3,3
1,"Robertson, Williams and Skinner",Than Campaign – LightPink,01/10/2022,Complete,Vanessa Cox,Design Issues,Normal,Design problem found in in footer Please addre...,Mobile,Chrome,https://www.thancampaign–lightpink.com,3,4
2,"Robertson, Williams and Skinner",Line Website – YellowGreen,07/02/2021,Complete,Diane Santiago,Copy Revision,Normal,Text correction required in area: footer This ...,Mobile,Firefox,https://www.linewebsite–yellowgreen.com,1,1
3,"Robertson, Williams and Skinner",Together Redesign – Red,19/12/2019,Complete,Ross Weaver,New Item,High,Build a new section in in footer Should be pri...,Mobile,Chrome,https://www.togetherredesign–red.com,8,4
4,Cooper PLC,Operation Website – FireBrick,23/07/2022,Complete,Larry Wood,Design Issues,Normal,Inconsistency in design of 'hero section' sect...,Mobile,Chrome,https://www.operationwebsite–firebrick.com,8,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Miller Group,Former Website – SlateGray,26/12/2021,Complete,Ryan Pollard,Requested Change,Urgent,Client wants an update to in hero section URGE...,Mobile,Chrome,https://www.formerwebsite–slategray.com,5,3
9996,Jenkins-Norris,Learn Website – HotPink,15/02/2023,Complete,Cassandra Clay,Design Issues,Normal,Improvement required for design in 'footer' se...,Mobile,Opera,https://www.learnwebsite–hotpink.com,3,4
9997,Garza and Sons,Future Dashboard – SaddleBrown,20/10/2022,Complete,Jacqueline Williams,New Item,Normal,Build a new section in in footer Please addres...,Mobile,Chrome,https://www.futuredashboard–saddlebrown.com,1,1
9998,Howard and Sons,Item Redesign – Green,11/05/2024,Complete,Lisa Smith,Design Issues,Normal,Layout issue detected in the about us page Ple...,Mobile,Chrome,https://www.itemredesign–green.com,5,4


In [3]:
# Número de proyectos por empresa
proyectos_por_empresa = df[['Company', 'Project Name']].drop_duplicates()
proyectos_count = proyectos_por_empresa.groupby('Company').size().reset_index(name='Num Projects')

# Número de requests por proyecto
requests_count = df.groupby(['Company', 'Project Name']).size().reset_index(name='Num Requests')

# Merge para unir ambos
combined = requests_count.merge(proyectos_count, on='Company')

# Mostrar ordenado por empresa y luego por cantidad de requests
combined.sort_values(['Num Projects', 'Num Requests'], ascending=[False, False])

,Company,Project Name,Num Requests,Num Projects
13,Brooks-Williams,Score Portal – Cornsilk,118,7
11,Brooks-Williams,Care Revamp – Tan,110,7
12,Brooks-Williams,Maybe Landing – DarkSlateBlue,108,7
14,Brooks-Williams,Strong Landing – Tan,108,7
10,Brooks-Williams,Big Redesign – Sienna,106,7
...,...,...,...,...
65,"Tucker, Shaffer and Figueroa",Exist Campaign – LightSkyBlue,167,1
70,Wheeler-Maddox,Difference Redesign – LightBlue,162,1
71,Williams-Williams,Speak Campaign – DarkRed,157,1
48,Mccall-Bright,Young Revamp – MediumBlue,156,1


In [10]:
#df2 = df[["Request Type" , "Status", "Input Date", "Requester", "Device", "Browser", "Request", "Page"]]

df.to_csv("synthetic_example.csv", index=False)

df.to_json("synthetic_example.json", orient="records", lines=True)